# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.3333333333333333` — пропуск один (`None` становится `NaN`). После `fillna(0)` получается `[1, 0, 3]`, среднее = 4 / 3 ≈ 1.33. Без заполнения было бы 2, так как `mean` пропускает `NaN`: ноль искажает среднее.
2. `2` — `strip` убирает пробелы, `lower` приводит к одному регистру: получается `['a', 'a', 'b']`, уникальных значений два.
3. `3` — результат `drop_duplicates()` не сохранён в `df`: метод возвращает новый DataFrame, а исходный не меняется.
4. `1005` — пробел удалён, `"1 000"` → `"1000"`, `to_numeric` переводит строки в числа: 1000 + 5 = 1005.
5. `[9, 10]` — первая дата в сентябре, вторая в октябре; `.dt.month` берёт номер месяца, время не влияет.

In [14]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [15]:
# 1
import pandas as pd

def clean(path):
    df = pd.read_csv(path)
    print('Прочитано:', len(df))


    df['amount'] = pd.to_numeric(df['amount'].astype(str).str.replace(' ', ''))
    print('После amount:', len(df))


    city_map = {
        'алматы': 'Алматы', 'almaty': 'Алматы',
        'астана': 'Астана', 'astana': 'Астана',
        'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
        'караганда': 'Караганда', 'актобе': 'Актобе', 'атырау': 'Атырау'}
    df['city'] = df['city'].str.strip().str.lower().map(city_map)
    print('После city:', len(df))


    df = df.drop_duplicates('order_id')
    print('После удаления дублей:', len(df))


    df['order_date'] = pd.to_datetime(df['order_date'])
    print('После дат:', len(df))


    df = df[~df['customer_id'].str.contains('TEST')]
    print('Без TEST:', len(df))

    return df

clean_df = clean('orders.csv')
print('Итого строк:', len(clean_df))



Прочитано: 5463
После amount: 5463
После city: 5463
После удаления дублей: 5283
После дат: 5283
Без TEST: 5280
Итого строк: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [16]:
# 2
sep = clean_df["order_date"] < "2026-10"
octo = ~sep
paid = clean_df['status'] == 'оплачен'

revenue_sep = clean_df.loc[sep & paid, 'amount'].sum()
revenue_oct = clean_df.loc[octo & paid, 'amount'].sum()
diff = (revenue_oct/revenue_sep - 1) * 100
print(f'Выручка за сентябрь : {revenue_sep}, выручка за октябрь : {revenue_oct}, изменение: {diff}')

for city in clean_df['city'].unique():
    c = clean_df['city'] == city
    r_sep = clean_df.loc[sep & paid & c, 'amount'].sum()
    r_oct = clean_df.loc[octo & paid & c, 'amount'].sum()
    d = (r_oct / r_sep - 1) * 100
    print(f'{city}: сентябрь {r_sep:,.0f}, октябрь {r_oct:,.0f}, изменение {d:+.1f}%')

#Сильнее всех падает Шымкент: −11,3%. На втором месте Караганда (−7,6%). Алматы (+1,0%) и Атырау (+5,0%) даже выросли.
#Да, совпадает: доли городов из таблицы «город × месяц» урока 4 (например, Шымкент 12,9% → 11,7%, Алматы 41,1% → 42,5%) получаются ровно из выручки, посчитанной здесь через pandas.
#Значит, очистка и расчёт в pandas дают тот же результат, что и в NumPy.

Выручка за сентябрь : 35243410.0, выручка за октябрь : 34419520.0, изменение: -2.337713632137184
Алматы: сентябрь 14,501,070, октябрь 14,643,020, изменение +1.0%
Астана: сентябрь 9,034,220, октябрь 8,850,530, изменение -2.0%
Караганда: сентябрь 3,648,520, октябрь 3,371,680, изменение -7.6%
Шымкент: сентябрь 4,558,650, октябрь 4,044,010, изменение -11.3%
Атырау: сентябрь 923,990, октябрь 969,860, изменение +5.0%
Актобе: сентябрь 2,576,960, октябрь 2,540,420, изменение -1.4%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [17]:
# 3
def drop_pct(d):
    dates = pd.to_datetime(d['order_date'], errors='coerce')
    amt = pd.to_numeric(d['amount'].str.replace(' ', ''), errors='coerce')
    paid = d['status'] == 'оплачен'
    sep = amt[paid & (dates.dt.month == 9)].sum()
    octo = amt[paid & (dates.dt.month == 10)].sum()
    return (octo / sep - 1) * 100

df = pd.read_csv('orders.csv')
print('Исходные:', len(df), f'{drop_pct(df):+.1f}%')

n = len(df)
df = df.drop_duplicates('order_id')
print('Дубли:', n - len(df), f'{drop_pct(df):+.1f}%')

n = len(df)
df = df[~df['customer_id'].str.contains('TEST')]
print('TEST:', n - len(df), f'{drop_pct(df):+.1f}%')

#Сильнее всего цифру исказили 3 тестовых заказа: после их удаления падение выручки сократилось с −19,6% до −2,3%.
#Реальное падение всего около 2%, остальное — тестовые записи в данных.

Исходные: 5463 -22.7%
Дубли: 180 -19.6%
TEST: 3 -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [18]:
# 4
nan_values = clean_df['delivery_days'].isna().sum()
by_city = clean_df.loc[clean_df['delivery_days'].isna(), 'city'].value_counts()
by_status = clean_df.loc[clean_df['delivery_days'].isna(), 'status'].value_counts()

shym = clean_df.loc[clean_df['city'] == 'Шымкент', 'delivery_days']

shym_a = shym.dropna()
a = (shym_a <= 2).mean()

shym_b = shym.fillna(shym.median())
b = (shym_b <= 2).mean()

shym_v = shym.fillna(0)
v = (shym_v <= 2).mean()

print(f'Пропуски в delivery_days: {nan_values}, в {by_city.index[0]} больше всего, в статусе {by_status.index[0]} больше всего')

print(f'(а) dropna: {a:.1%}, (б) медиана: {b:.1%}, (в) ноль: {v:.1%}')
# Честный (а) dropna. Мы считаем долю только по заказам, у которых срок доставки известен, и ничего не выдумываем.
# (б) медиана подставляет выдуманное «типичное» значение (больше 2 дней),
#  все пропуски автоматически становятся «медленными», и доля занижается (29.0%).
# (в) ноль делает вид, что заказы доставлены мгновенно,
#  все пропуски попадают в «за 2 дня», и доля завышается (31.8%).


Пропуски в delivery_days: 158, в Алматы больше всего, в статусе оплачен больше всего
(а) dropna: 29.8%, (б) медиана: 29.0%, (в) ноль: 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [19]:
#5
df = pd.read_csv('orders.csv')
df = df.drop_duplicates('order_id')

city_map = {
    'алматы': 'Алматы', 'almaty': 'Алматы',
    'астана': 'Астана', 'astana': 'Астана',
    'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
    'караганда': 'Караганда', 'актобе': 'Актобе', 'атырау': 'Атырау'}
df['city'] = df['city'].str.strip().str.lower().map(city_map)

print(df['city'].nunique(), len(df))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [20]:
# 6
counts = clean_df['order_date'].dt.day_name().value_counts()
print(counts)

max_day, max_val = counts.idxmax(), counts.max()
min_day, min_val = counts.idxmin(), counts.min()
diff = (1 - min_val / max_val) * 100

print(f'Меньше всего заказов в {min_day} ({min_val}), больше всего в {max_day} ({max_val})')
print(f'В {min_day} заказов на {diff:.1f}% меньше')
# Меньше всего заказов в понедельник (684), больше всего в четверг (786): разница 13%. Со вторника по субботу нагрузка почти одинаковая (773–786),
# а в воскресенье и понедельник заметно ниже.
# Значит, в эти два дня можно ставить чуть меньше курьеров, но сильно сокращать смены не стоит: разница умеренная.


order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего заказов в Monday (684), больше всего в Thursday (786)
В Monday заказов на 13.0% меньше


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [21]:
# 7

shym = clean_df[(clean_df['city'] == 'Шымкент') & clean_df['delivery_days'].notna()]


print('Сентябрь:', shym.loc[shym['order_date'].dt.month == 9, 'delivery_days'].mean())
print('Октябрь:', shym.loc[shym['order_date'].dt.month == 10, 'delivery_days'].mean())


fast = shym[shym['delivery_days'] < 5].sort_values('order_date')
last_fast = fast['order_date'].iloc[-1]
print('Последняя быстрая доставка:', last_fast)

break_date = shym.loc[shym['order_date'] > last_fast, 'order_date'].sort_values().iloc[0]
print('Доставка сломалась с:', break_date)

other = clean_df[clean_df['city'] != 'Шымкент']
before = other[other['order_date'] < break_date].groupby('city')['delivery_days'].mean()
after = other[other['order_date'] >= break_date].groupby('city')['delivery_days'].mean()
print(pd.DataFrame({'до': before, 'после': after}))

# С 1 октября 2026 года в Шымкенте нет ни одной доставки быстрее 5 дней: средний срок вырос с 2.0 дня в сентябре до 6.5 дня в октябре, примерно в 3 раза.
# В других городах срок остался около 2 дней, значит, проблема только в Шымкенте.
# В первую очередь проверить, что изменилось там 1 октября: курьерская служба, склад или число курьеров.

Сентябрь: 2.031518624641834
Октябрь: 6.5358024691358025
Последняя быстрая доставка: 2026-09-30 22:00:00
Доставка сломалась с: 2026-10-01 10:00:00
                 до     после
city                         
Актобе     2.042553  2.097436
Алматы     1.980218  2.013436
Астана     2.043143  1.995385
Атырау     2.060606  2.013889
Караганда  1.896552  1.991453


## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [22]:
# 8
def to_amount(s):
    s = (s.astype(str)
          .str.replace(' ', '')
          .str.replace('\xa0', '')
          .str.replace(',', '.'))
    return pd.to_numeric(s, errors='coerce')

print(to_amount(pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])))

raw = pd.read_csv('orders.csv')
print('Сумма amount:', to_amount(raw['amount']).sum())

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Сумма amount: 109883360.0
